In [3]:
from clustering.clustering import create_cluster
import polars as pl
from genai.mistral import get_mistral_cluster_tags, ClusterWithTags
import time

folder = "ribier_v1"
df = pl.read_parquet(f"./books_work/{folder}/data/{folder}.parquet")
docs = df["embedding"].to_list()
print("Creating clusters...")
cluster, soft, samples, iters = create_cluster(docs, run_num=1)
df = df.with_columns(
    pl.Series("cluster", cluster),
    pl.Series("soft_cluster", soft),
)
print("Calculating cluster counts and percentages...")
n = len(df)
cluster_counts = df["cluster"].value_counts().sort("cluster")
cluster_counts = cluster_counts.with_columns(cluster_perc=cluster_counts["count"] / n)
clusters_to_sub_df = cluster_counts.filter(pl.col("count") ** (1 / 3) >= 3)
clusters_to_sub = clusters_to_sub_df["cluster"].to_list()
df = df.with_columns(sub_cluster=pl.lit(-1).cast(pl.Int64))
df = df.with_columns(sub_labels=pl.lit(None).cast(pl.List(pl.Utf8)))

print(f"Clusters to sub-cluster: {clusters_to_sub}")
# Let's get sub-clusters for clusters that have enough samples, and get their tags using Mistral
for c in clusters_to_sub:
    cluster_count = clusters_to_sub_df.filter(pl.col("cluster") == c)["count"][0] ** (
        1 / 3
    )
    cluster_count = int(cluster_count)
    sub_df = df.filter(pl.col("cluster") == c)
    sub_docs = sub_df["embedding"].to_list()
    sub_cluster, _, sub_samples, _ = create_cluster(
        sub_docs, run_num=1, soft_assign=False, num_clusters=cluster_count
    )
    # Get sub-cluster tags using Mistral
    print(f"Getting tags for sub-cluster {c}...")
    sample_docs: list[ClusterWithTags] = [
        ClusterWithTags(label=i, tags=df[s].select("markdown").to_series().to_list())
        for i, s in enumerate(sub_samples)
    ]
    res = get_mistral_cluster_tags(sample_docs)
    sub_samples_dict = {item.label: item.tags for item in res}
    sub_samples_list = [sub_samples_dict.get(c) for c in sub_cluster]

    # Assign sub_cluster to sub_df
    sub_df = sub_df.with_columns(pl.Series("sub_c", sub_cluster))
    sub_df = sub_df.with_columns(pl.Series("sub_s", sub_samples_list))

    df = df.join(
        sub_df.select(["chunk_id", "sub_c", "sub_s"]), on="chunk_id", how="left"
    )
    df = df.with_columns(
        sub_cluster=(
            pl.when(pl.col("sub_c").is_not_null())
            .then(pl.col("sub_c"))
            .otherwise(pl.col("sub_cluster"))
        ),
        sub_labels=(
            pl.when(pl.col("sub_s").is_not_null())
            .then(pl.col("sub_s"))
            .otherwise(pl.col("sub_labels"))
        ),
    )
    df = df.drop("sub_c")
    df = df.drop("sub_s")
    # If you want to fill nulls with a default value (e.g., -1), uncomment:
    df = df.with_columns(pl.col("sub_cluster").fill_null(-1))

# Now let's get the cluster tags for the main clusters using Mistral
sample_docs: list[ClusterWithTags] = [
    ClusterWithTags(label=i, tags=df[s].select("markdown").to_series().to_list())
    for i, s in enumerate(samples)
]

res = get_mistral_cluster_tags(sample_docs)
res_dict = {item.label: item.tags for item in res}
res_list = [res_dict.get(c) for c in cluster]
df = df.with_columns(pl.Series("cluster_tags", res_list))
df

Creating clusters...
  Estimated number of clusters: 14
  Restart 1/10 (seed 10) — score: 0.9375
  Restart 2/10 (seed 11) — score: 0.9371
  Restart 3/10 (seed 12) — score: 0.9384
  Restart 4/10 (seed 13) — score: 0.9376
  Restart 5/10 (seed 14) — score: 0.9375
  Restart 6/10 (seed 15) — score: 0.9376
  Restart 7/10 (seed 16) — score: 0.9374
  Restart 8/10 (seed 17) — score: 0.9372
  Restart 9/10 (seed 18) — score: 0.9374
  Restart 10/10 (seed 19) — score: 0.9379
  Best score: 0.9384
Calculating cluster counts and percentages...
Clusters to sub-cluster: [0, 2, 12]
  Restart 1/10 (seed 10) — score: 0.9430
  Restart 2/10 (seed 11) — score: 0.9433
  Restart 3/10 (seed 12) — score: 0.9436
  Restart 4/10 (seed 13) — score: 0.9439
  Restart 5/10 (seed 14) — score: 0.9434
  Restart 6/10 (seed 15) — score: 0.9437
  Restart 7/10 (seed 16) — score: 0.9447
  Restart 8/10 (seed 17) — score: 0.9430
  Restart 9/10 (seed 18) — score: 0.9442
  Restart 10/10 (seed 19) — score: 0.9431
  Best score: 0.944

letter_id,id,markdown,pdf_pages,title,chunk_id,word_length,full_markdown,embedding,cluster,soft_cluster,sub_cluster,sub_labels,cluster_tags
u32,u32,str,list[i64],str,f64,i64,str,list[f64],i64,list[i64],i64,list[str],list[str]
10000,1000,"""LE PAPE PAVL III. AV ROT FRANC…",[46],"""LE PAPE PAVL III. AV ROT FRANC…",10000.0,176,"""LE PAPE PAVL III. AV ROT FRANC…","[0.024414, 0.036377, … -0.024536]",10,[10],-1,null,"[""PapalConciliarDiplomacy"", ""RhetoricOfPapalAuthority"", … ""PrincelyMediationEfforts""]"
10001,1001,"""LETTRE DE MONLVC AV CARDINAL D…","[46, 47]","""LETTRE DE MONLVC AV CARDINAL D…",10001.0,470,"""LETTRE DE MONLVC AV CARDINAL D…","[-0.00296, 0.011841, … -0.024292]",2,"[2, 9, 12]",3,"[""PapalMediationAttempts"", ""SuspicionOfDividedLoyalties"", … ""TruceNegotiations""]","[""PapalDiplomacy"", ""NeutralityStrategies"", … ""FrenchMediation""]"
10002,1002,"""LETTRE DES AVOTERS ET CONSEILL…","[47, 48]","""LETTRE DES AVOTERS ET CONSEILL…",10002.0,452,"""LETTRE DES AVOTERS ET CONSEILL…","[-0.0354, -0.02771, … 0.023071]",12,"[2, 9, 12]",0,"[""DiplomaticNegotiations"", ""SuzerainVassalTensions"", … ""StateSovereigntyClaims""]","[""DiplomaticEspionage"", ""RenaissanceStatecraft"", … ""CourtlyIntrigue""]"
10003,1003,"""REMARQUES SVR LADITE VILLE DE …","[48, 49, … 54]","""REMARQUES SVR LADITE VILLE DE …",10003.0,1878,"""REMARQUES SVR LADITE VILLE DE …","[-0.036621, -0.010559, … -0.013855]",8,[8],-1,null,"[""DiplomaticNegotiations"", ""ConfessionalFragmentation"", … ""ReligiousDissent""]"
10003,1003,"""Cethol. 1192. 6 commiæer Janoe…","[48, 49, … 54]","""REMARQUES SVR LADITE VILLE DE …",10003.1,778,"""REMARQUES SVR LADITE VILLE DE …","[0.007721, -0.031738, … -0.024902]",10,"[10, 13]",-1,null,"[""PapalConciliarDiplomacy"", ""RhetoricOfPapalAuthority"", … ""PrincelyMediationEfforts""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…
10261,1261,"""DE L'AUBESPINE. --- ## LE ROY…","[656, 657]","""DE L'AUBESPINE.""",10261.0,432,"""DE L'AUBESPINE. --- ## LE ROY…","[-0.04541, -0.05542, … 0.011902]",1,[1],-1,null,"[""GermanPrincesResistance"", ""ElectoralAlliances"", … ""OttomanThreatPerception""]"
10262,1262,"""# A MONSIEUR LE DAVHIN. Le Ca…","[657, 658]","""A MONSIEUR LE DAVHIN.""",10262.0,498,"""# A MONSIEUR LE DAVHIN. Le Ca…","[-0.004944, 0.012268, … -0.06543]",2,"[0, 2, 3]",3,"[""PapalMediationAttempts"", ""SuspicionOfDividedLoyalties"", … ""TruceNegotiations""]","[""PapalDiplomacy"", ""NeutralityStrategies"", … ""FrenchMediation""]"
10263,1263,"""LE CARDINAL DE BOLOGNE ## AV R…","[658, 659]","""LE CARDINAL DE BOLOGNE AV ROT…",10263.0,560,"""LE CARDINAL DE BOLOGNE ## AV R…","[-0.036133, -0.028687, … -0.050781]",0,[0],0,"[""PapalDiplomacy"", ""FrancoImperialRivalry"", … ""MediterraneanCrusadingZeal""]","[""DiplomaticAlliances"", ""AntiHabsburgCoalition"", … ""GeopoliticalRivalry""]"


In [4]:
df.write_parquet(f"./books_work/ribier_v1/data/ribier_v1.parquet")

In [5]:
import polars as pl

df = pl.read_parquet(f"./books_work/ribier_v1/data/ribier_v1.parquet")
df.head()

letter_id,id,markdown,pdf_pages,title,chunk_id,word_length,full_markdown,embedding,cluster,soft_cluster,sub_cluster,sub_labels,cluster_tags
u32,u32,str,list[i64],str,f64,i64,str,list[f64],i64,list[i64],i64,list[str],list[str]
10000,1000,"""LE PAPE PAVL III. AV ROT FRANC…",[46],"""LE PAPE PAVL III. AV ROT FRANC…",10000.0,176,"""LE PAPE PAVL III. AV ROT FRANC…","[0.024414, 0.036377, … -0.024536]",10,[10],-1,null,"[""PapalConciliarDiplomacy"", ""RhetoricOfPapalAuthority"", … ""PrincelyMediationEfforts""]"
10001,1001,"""LETTRE DE MONLVC AV CARDINAL D…","[46, 47]","""LETTRE DE MONLVC AV CARDINAL D…",10001.0,470,"""LETTRE DE MONLVC AV CARDINAL D…","[-0.00296, 0.011841, … -0.024292]",2,"[2, 9, 12]",3,"[""PapalMediationAttempts"", ""SuspicionOfDividedLoyalties"", … ""TruceNegotiations""]","[""PapalDiplomacy"", ""NeutralityStrategies"", … ""FrenchMediation""]"
10002,1002,"""LETTRE DES AVOTERS ET CONSEILL…","[47, 48]","""LETTRE DES AVOTERS ET CONSEILL…",10002.0,452,"""LETTRE DES AVOTERS ET CONSEILL…","[-0.0354, -0.02771, … 0.023071]",12,"[2, 9, 12]",0,"[""DiplomaticNegotiations"", ""SuzerainVassalTensions"", … ""StateSovereigntyClaims""]","[""DiplomaticEspionage"", ""RenaissanceStatecraft"", … ""CourtlyIntrigue""]"
10003,1003,"""REMARQUES SVR LADITE VILLE DE …","[48, 49, … 54]","""REMARQUES SVR LADITE VILLE DE …",10003.0,1878,"""REMARQUES SVR LADITE VILLE DE …","[-0.036621, -0.010559, … -0.013855]",8,[8],-1,null,"[""DiplomaticNegotiations"", ""ConfessionalFragmentation"", … ""ReligiousDissent""]"
10003,1003,"""Cethol. 1192. 6 commiæer Janoe…","[48, 49, … 54]","""REMARQUES SVR LADITE VILLE DE …",10003.1,778,"""REMARQUES SVR LADITE VILLE DE …","[0.007721, -0.031738, … -0.024902]",10,"[10, 13]",-1,null,"[""PapalConciliarDiplomacy"", ""RhetoricOfPapalAuthority"", … ""PrincelyMediationEfforts""]"


In [6]:
with pl.Config(tbl_rows=26):
    display(df["cluster"].value_counts().sort("cluster"))

cluster,count
i64,u32
0,40
1,17
2,97
3,21
4,14
5,18
6,4
7,14
8,7


In [5]:
cluster13_samples = (
    df.filter(pl.col("cluster1") == 13)
    .select("markdown")
    .sample(5)
    .to_series()
    .to_list()
)
cluster13_samples

["\n\nJe renferme dans ce Livre le refe du Regne de François I. c'est à dire depuis 1541. jusqu'en 1547. au mois de Mars ; n'ayant pas affex de pieces pour faire vn Livre de chaque année, comme nous avons commencé. La disgrace de Mr. le Connesable en est la cause: car comme les Negotiations des Ambassadors et autres Affaires ne luy furent plus addresfées et ont passé en d'autres mains, se n'en ay pû recouurer les Puses Originales dans les Maisons anciennes où j'ay trouvé les Precedentes sous François I. et les faisantes sous Henry II. J's semble que le Roy pendant la disgrace Mr. le Connesable, pris luy-même le jour des Affaires. Neantmoins Mr. l'Admiral fut comme le Fanory, et après luy ceux qui eurent le plus de credit, même dans les affaires d'Etat, furent les Srs. Bayard et Bochete tous deux Secretaires des Finances. Aussi furent-ils éloignés de la Cour, et même ledit Bayard rudement traité sous Henry II. lors que le Connesable fut ressabby en faveur.\n\nLes Secretaires d'Etat en c

In [7]:
cluster9_sampels = (
    df.filter(pl.col("cluster1") == 9)
    .select("markdown")
    .sample(5)
    .to_series()
    .to_list()
)
cluster9_sampels

["L'Etat d'Aurances que le Roy encoye présentement par doutes, l'Empereur, luy fera les très affectueuses &amp; très cordiales recommandations dudit Sr. Et aptes luy auoit présenté les Lettres de Créance qu'il luy écrit, luy dira qu'étant naguères retourné le Sr. César Canteline, cy-deuant dépêché pour le fait de la Treue générale entre la Chreštientė &amp; le Turc ; ledit Sr. Roy pour toujours correspondre à la bonne &amp; parfaite amitié que Dieu a mise &amp; rétablie entre l'Empereur &amp; luy, a bien voulu l'adulser &amp; faire participant de tout ce qui a été négotié envers le Turc, en ce qui concernoit le fait de ladite Treue générale. Et pour luy faire entendre clairement &amp; à la vérité comme les choses s'y font passées; c'est qu'à Partiuée dudit Canteline p. d'duets le Sr. Rincon, il luy décera entièrement la charge qu'il auoit, &amp; l'occasion pour laquelle il eftoit là envoyé : &amp; ce fait, ils se transporteront ensemble à la Porte dudit Turc, auquel Rincon suiuant l'in

In [9]:
cluster12_samples = (
    df.filter(pl.col("cluster1") == 12)
    .select("markdown")
    .sample(5)
    .to_series()
    .to_list()
)
cluster12_samples

["\n\nD I L E C T E fili, salutem &amp; Apostolica m benedictionem. Audito Christianiflimi Regis sapienti judicio, quo nobilitatem tuam ad supremum Regni Francie Magistratum evexit, magna profecto latitia affecti sumus. Quot viro &amp; probo, &amp; prudenti, &amp; Q. iij\n\nOriginal du 11. Mars.3\n\nLe ferey voir cyapres, par divers les Lettres, que les Papes traitent les Ducs même fou. Certains de Nobiliu vis &amp; de Nobilius.\n\nLes Eloges du Pape &amp; du Connoissable se veront cyapres.\n\nOriginal.\n\n\n\n# LETTRES ET MEMOIRES DESTAT.\n\nmultis virtutibus ornato, ex potestas accellerit, quâ &amp; rei Christianae, &amp; Apostolicae Sedis possit maiori multo auctoritate patrocinari. Qualem mentem, animum qu in omnibus actionibus suis temper ostendit. Graculamur itaq; primum Regia Majestati, quæ beneficiu dando accepit, cum digno dedit. Deinde Nobilitati tuæ, quæ tantum Principis iudicium in se promeruit. Er speramus hoc, Deo benevolente atq; auctore, factum esse, vit ea Fidelium rem

In [7]:
from clustering.clustering import create_cluster
import polars as pl

folder = "ribier_v1"

df1 = pl.read_parquet(f"./books_work/{folder}/data/{folder}_chunked.parquet")
docs = df1["embedding"].to_list()
cluster3, soft3, samples3, iters3 = create_cluster(docs, run_num=1, soft_assign=False)
cluster4, soft4, samples4, iters4 = create_cluster(docs, run_num=2, soft_assign=False)
df1 = df1.with_columns(cluster3=pl.Series(cluster3), cluster4=pl.Series(cluster4))
print("Cluster 3:")
with pl.Config(tbl_rows=25):
    display(df1["cluster3"].value_counts().sort("cluster3"))
print("Cluster 4:")
with pl.Config(tbl_rows=25):
    display(df1["cluster4"].value_counts().sort("cluster4"))

  Estimated number of clusters: 14
  Restart 1/10 (seed 10) — score: 0.9402
  Restart 2/10 (seed 11) — score: 0.9400
  Restart 3/10 (seed 12) — score: 0.9403
  Restart 4/10 (seed 13) — score: 0.9403
  Restart 5/10 (seed 14) — score: 0.9398
  Restart 6/10 (seed 15) — score: 0.9402
  Restart 7/10 (seed 16) — score: 0.9404
  Restart 8/10 (seed 17) — score: 0.9399
  Restart 9/10 (seed 18) — score: 0.9401
  Restart 10/10 (seed 19) — score: 0.9393
  Best score: 0.9404
  Estimated number of clusters: 14
  Restart 1/10 (seed 20) — score: 0.9406
  Restart 2/10 (seed 21) — score: 0.9400
  Restart 3/10 (seed 22) — score: 0.9396
  Restart 4/10 (seed 23) — score: 0.9399
  Restart 5/10 (seed 24) — score: 0.9401
  Restart 6/10 (seed 25) — score: 0.9394
  Restart 7/10 (seed 26) — score: 0.9403
  Restart 8/10 (seed 27) — score: 0.9402
  Restart 9/10 (seed 28) — score: 0.9386
  Restart 10/10 (seed 29) — score: 0.9399
  Best score: 0.9406
Cluster 3:


cluster3,count
i64,u32
0,75
1,21
2,17
3,78
4,6
5,26
6,23
7,9
8,3


Cluster 4:


cluster4,count
i64,u32
0,28
1,65
2,19
3,9
4,23
5,6
6,37
7,68
8,11


In [11]:
df.sample(5)

letter_id,id,title,pdf_pages,book_pages,markdown,chunk_id,word_length,full_markdown,embedding,cluster1,cluster2
u32,u32,str,list[i64],list[str],str,f64,i64,str,list[f64],i64,i64
10129,10129,"""APOLOGIE""","[327, 328, … 356]","[""104"", ""105"", … ""333""]","""Nous en avons des témoignages …",10129.6,1863,""" des permissions de Duel obte…","[-0.043457, -0.029053, … -0.014771]",8,11
10082,10082,"""LA CHAMBRE DES COMPTES.""","[226, 227]","[""103"", ""204""]",""" Elle depute tant vers le Roy…",10082.0,146,""" Elle depute tant vers le Roy…","[-0.014221, -0.008301, … -0.027832]",12,8
10050,10050,"""AVDIT CONNESTABLE""",[174],[],""" J. Jacques Castron Commis à …",10050.0,64,""" J. Jacques Castron Commis à …","[-0.025146, -0.019775, … 0.001678]",4,0
10022,10022,"""AVTRES ARTICLES DÉBATTVS""","[82, 83, … 85]","[""55"", ""63"", ""2""]",""" en ladite communication non …",10022.0,1570,""" en ladite communication non …","[-0.032715, 0.003464, … 0.00589]",4,11
10271,10271,"""CAROLVS.""",[568],"[""547""]",""" """,10271.0,0,""" ""","[-0.048584, -0.026123, … -0.038818]",1,2


letter_id,id,title,pdf_pages,book_pages,markdown,chunk_id,word_length,full_markdown,embedding,cluster1,cluster2,subcluster1
u32,u32,str,list[i64],list[str],str,f64,i64,str,list[f64],i64,i64,null
10262,10262,"""AV CARDINAL DV BELLAY.""",[554],"[""533""]",""" Sur quelques intrigues de Co…",10262.0,345,""" Sur quelques intrigues de Co…","[-0.047852, -0.026733, … 0.021362]",12,8,null
10292,10292,"""GOSTAVVS.""","[592, 593, … 595]","[""571"", ""52"", … ""374""]",""" Cccc iij52 LETTRES ET MEMOIR…",10292.0,1490,""" Cccc iij52 LETTRES ET MEMOIR…","[-0.044434, -0.009094, … -0.026611]",13,8,null
10009,10009,"""PAVLVS PAPA III.""",[46],"[""15""]",""" CHARISSIME in Christo Fili n…",10009.0,142,""" CHARISSIME in Christo Fili n…","[0.00531, 0.01001, … -0.019653]",11,6,null
10029,10029,"""REMARQUES SVR LEDIT SIEVR""","[116, 117, … 120]","[""23"", ""34"", … ""99""]","""de Selve Ambaffadeur. I'A cy-…",10029.0,1992,"""de Selve Ambaffadeur. I'A cy-…","[-0.001022, -0.030273, … 0.020264]",9,7,null
10006,10006,"""ENSVIT LEDIT PLAIDOYE.""","[27, 28, … 39]","[""6"", ""7"", … ""13""]","""Sire, Varus Orateur insigne vo…",10006.0,1761,""" Sire, Varus Orateur insigne …","[-0.039062, -0.052002, … -0.034424]",12,11,null
